In [7]:
import numpy as np
import pandas as pd
import re

In [4]:
df=pd.read_csv('C:/Users/X/Desktop/patients.csv')
df

,patient_id,dob,sex,zip,enrollment_start,enrollment_end
0,1,1980/01/05,M,2139,2021-01-01,2021-12-31
1,2,05-14-1975,Female,2139,2021/02/01,2021-11-30
2,3,1979-13-01,F,2140,2021-03-01,2021-12-31
3,3,1979-01-13,F,2140,2021-03-01,2021-12-31
4,4,NaN,U,2141,2021-01-15,2021-10-10


In [ ]:
#task 1:Standardize patient_id to 3-digit character IDs.
def clean_patient_id(x):
    if pd.isna(x):
        return np.nan
    #re.sub(pattern, replacement, from this text)
    digits = re.sub(r"\D", "", str(x))
    #zfill() is a string method that pads the left side with zeros until the string reaches a given length. In this case length=3
    return digits.zfill(3) if digits else np.nan
    #nan is np.nan = “Not a Number”
df["patient_id"] = df["patient_id"].apply(clean_patient_id) #Take each value in the patient_id column and pass it into the function clean_patient_id
df

,patient_id,dob,sex,zip,enrollment_start,enrollment_end
0,001,1980/01/05,M,2139,2021-01-01,2021-12-31
1,002,05-14-1975,Female,2139,2021/02/01,2021-11-30
2,003,1979-13-01,F,2140,2021-03-01,2021-12-31
3,003,1979-01-13,F,2140,2021-03-01,2021-12-31
4,004,NaN,U,2141,2021-01-15,2021-10-10


In [ ]:
#task 2:Parse dates robustly (ymd, mdy, etc.), flag impossible dates.


,patient_id,dob,sex,zip,enrollment_start,enrollment_end
0,001,1980/01/05,M,2139,2021-01-01,2021-12-31
1,002,05-14-1975,Female,2139,2021/02/01,2021-11-30
2,003,1979-13-01,F,2140,2021-03-01,2021-12-31
3,003,1979-01-13,F,2140,2021-03-01,2021-12-31
4,004,NaN,U,2141,2021-01-15,2021-10-10


In [18]:
today = pd.Timestamp.today().normalize() 

In [19]:
def parse_mixed_date(series: pd.Series) -> pd.Series:
    """Robust date parse for mixed formats; invalid => NaT."""
    # pandas handles many formats; coerce invalid to NaT
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True).dt.normalize()
df["dob"] = parse_mixed_date(df["dob"])
df["invalid_dob"] = df["dob"].isna() | (df["dob"] > today) | (df["dob"] < pd.Timestamp("1900-01-01"))
df['invalid_dob']

C:\Users\X\AppData\Local\Temp\ipykernel_26272\1747129096.py:4: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series, errors="coerce", infer_datetime_format=True).dt.normalize()


0    False
1     True
2     True
3     True
4     True
Name: invalid_dob, dtype: bool

In [20]:
df

,patient_id,dob,sex,zip,enrollment_start,enrollment_end,invalid_dob
0,001,1980-01-05,M,2139,2021-01-01,2021-12-31,False
1,002,NaT,Female,2139,2021/02/01,2021-11-30,True
2,003,NaT,F,2140,2021-03-01,2021-12-31,True
3,003,NaT,F,2140,2021-03-01,2021-12-31,True
4,004,NaT,U,2141,2021-01-15,2021-10-10,True


In [23]:
#remove duplicates
dedup_cols = ["patient_id", "dob", "sex", "zip", "enrollment_start", "enrollment_end"]
df = df.drop_duplicates(subset=dedup_cols, keep="first").copy()
df

,patient_id,dob,sex,zip,enrollment_start,enrollment_end,invalid_dob
0,001,1980-01-05,M,2139,2021-01-01,2021-12-31,False
1,002,NaT,Female,2139,2021/02/01,2021-11-30,True
2,003,NaT,F,2140,2021-03-01,2021-12-31,True
4,004,NaT,U,2141,2021-01-15,2021-10-10,True
